# KrVoiceAI · Colab GPU Worker

这台 Colab 当 **GPU worker**:连 `krvoice.cicy-ai.com` 的任务队列,拉任务 → GPU 出片 → 回传成片。

**用户在口播工坊(krvoice.cicy-ai.com)提交任务,这里自动领走出片。** 素材从 OSS 拉,不需要 Google Drive。

**先选 GPU**:`代码执行程序 → 更改运行时类型 → A100 或 L4`(LatentSync 512 需 24G+,L4/A100 都行,T4 不够)。然后从上到下逐格运行。

In [ ]:
# 1. 确认 GPU(A100 或 L4,显存 ≥23G)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 拉最新代码
%cd /content
import os
if os.path.isdir('KrVoiceAI'):
    %cd KrVoiceAI
    !git pull -q
else:
    !git clone -q https://github.com/cicy-ai/KrVoiceAI.git
    %cd KrVoiceAI
!git log --oneline -1

In [ ]:
# 3. 起队列 worker —— 拉 krvoice.cicy-ai.com 的任务 → 出片 → 回传。后台常驻(subprocess,cell 结束不被杀)。
import os, time, subprocess
subprocess.run(['pkill', '-f', 'pull_worker.py'])
time.sleep(1)
env = dict(os.environ, QUEUE_URL='https://krvoice.cicy-ai.com')
subprocess.Popen(['python', 'colab/pull_worker.py'], cwd='/content/KrVoiceAI', env=env,
                 stdout=open('/content/pull_worker.log', 'w'), stderr=subprocess.STDOUT,
                 start_new_session=True)
time.sleep(10)
print(open('/content/pull_worker.log').read()[-900:] if os.path.exists('/content/pull_worker.log') else '(worker 启动中,重跑第4格看日志)')

In [ ]:
# 4. 看 worker 状态 / 出片进度(反复运行本格)
import os
print(open('/content/pull_worker.log').read()[-2500:] if os.path.exists('/content/pull_worker.log') else '(worker 未启动,先跑第3格)')

---
- worker 起来后会每几秒轮询队列;用户在口播工坊点「生成」→ 这里自动领任务出片。
- 改代码后:重跑**第2格**(git pull)+ **第3格**(重启 worker)。
- 页面刷新/断连不影响已在跑的 worker(setsid 脱离);运行时被回收才会停。